In [61]:
#!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [62]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()


print(f"Length of dataset in characters: {len(text):,}")


batch_size = 64
context_size = 256
max_iters = 5000
eval_interval = 300
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embedding_dim = 384
dropout = 0.2
num_heads = 6  # 384/6 = 64 dim per head
n_layers = 6

print (f"Using device: {device}")


Length of dataset in characters: 1,115,394
Using device: cuda


In [63]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(chars)
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


In [64]:
# map chars to ints

s_to_i = {ch: i for i, ch in enumerate(chars)}
i_to_s = {i: ch for i, ch in enumerate(chars)}


def encode(s: str) -> list[int]:
    ls = [s_to_i[c] for c in s]
    return ls


def decode(ls: list[int]) -> str:
    st = [i_to_s[i] for i in ls]
    st = "".join(st)
    return st


test_str = "She dont believe in shooting stars"

print(encode(test_str))
print(decode(encode(test_str)))

[31, 46, 43, 1, 42, 53, 52, 58, 1, 40, 43, 50, 47, 43, 60, 43, 1, 47, 52, 1, 57, 46, 53, 53, 58, 47, 52, 45, 1, 57, 58, 39, 56, 57]
She dont believe in shooting stars


In [65]:
data = torch.tensor(encode(text), dtype=torch.long)

print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [66]:
# train val
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [67]:
context_len = 8
train_data[: context_len + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [68]:
x = train_data[:context_len]
y = train_data[1 : context_len + 1]
for t in range(context_len):
    context = x[: t + 1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [69]:
def get_batch(split: str) -> tuple:
    data = train_data if split == "train" else val_data

    # random context size size data from data
    ix = torch.randint(len(data) - context_size, (batch_size,))

    # stack of 1d tensors for inputs and targets, they become a batch_size x context_size matrix (completely indpendent, just for eficiency )
    x = torch.stack([data[i : i + context_size] for i in ix])
    y = torch.stack([data[i + 1 : i + context_size + 1] for i in ix])
    
    x, y = x.to(device), y.to(device)
    
    return x, y


xb, yb = get_batch("train")
print("inputs: ")
print(xb.shape)
print(xb)

print("targets: ")
print(yb.shape)
print(yb)

print("---------")

for b in range(batch_size):  # batch dimension
    for t in range(context_size):  # time dimension
        context = xb[b, : t + 1]
        target = yb[b, t]
        #print(f"when input is {context.tolist()} the target: {target}")

inputs: 
torch.Size([64, 256])
tensor([[ 0, 26, 53,  ..., 56, 43, 47],
        [60, 43, 56,  ..., 56,  1, 41],
        [26, 21, 33,  ..., 26, 21, 13],
        ...,
        [ 5, 57,  1,  ...,  1, 35, 47],
        [56, 53, 53,  ..., 59, 50, 42],
        [42, 47, 56,  ..., 39, 56,  1]], device='cuda:0')
targets: 
torch.Size([64, 256])
tensor([[26, 53, 58,  ..., 43, 47, 45],
        [43, 56,  1,  ...,  1, 41, 53],
        [21, 33, 31,  ..., 21, 13, 10],
        ...,
        [57,  1, 52,  ..., 35, 47, 50],
        [53, 53, 58,  ..., 50, 42,  1],
        [47, 56, 43,  ..., 56,  1, 51]], device='cuda:0')
---------


In [70]:
class Head(nn.Module):
    """ one head of self attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embedding_dim, head_size, bias=False)
        self.query = nn.Linear(n_embedding_dim, head_size, bias=False)
        self.value = nn.Linear(n_embedding_dim, head_size, bias=False)
        self.register_buffer("trill", torch.tril(torch.ones(context_size, context_size)))
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)  # (B,T,C)
        q = self.query(x)  # (B,T,C)
        
        # compute attention scores --- (B,T,C) @ (B,C,T) --> (B,T,T)
        wei = q @ k.transpose(-2, -1) * C**-0.5 
        wei= wei.masked_fill(self.trill[:T, :T] == 0, float("-inf")) 
        wei = F.softmax(wei, dim=-1) 
        wei = self.dropout(wei)
        v = self.value(x)  # (B,T,C)
        out = wei @ v  # (B,T,T) @ (B,T,C) --> (B,T,C)
        return out

In [71]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embedding_dim, n_embedding_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

In [72]:
class FeedForward(nn.Module):
    
    def __init__(self, n_embedding_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embedding_dim, 4*n_embedding_dim),
            nn.ReLU(),
            nn.Linear(4*n_embedding_dim, n_embedding_dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        return self.net(x)

In [73]:
class TransformerBlock(nn.Module):
    
    def __init__(self, n_embedding_dim, num_heads):
        super().__init__()
        head_size = n_embedding_dim // num_heads
        self.sa_heads = MultiHeadAttention(num_heads, head_size) #4 heads of 8 dim self attention
        self.ffwd = FeedForward(n_embedding_dim)
        self.ln1 = nn.LayerNorm(n_embedding_dim)
        self.ln2 = nn.LayerNorm(n_embedding_dim)
        
        
    def forward(self, x):
        x = x + self.sa_heads(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [74]:

class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embedding_dim)
        self.position_embedding_table = nn.Embedding(context_size, n_embedding_dim)
        self.transformer_blocks = nn.Sequential(*[TransformerBlock(n_embedding_dim, num_heads=num_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(n_embedding_dim)
        self.lm_head = nn.Linear(n_embedding_dim, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensors of integers

        B,T = idx.shape
        
        tok_emb = self.token_embedding_table(idx) # (B,T,C) 
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        tok_emb = tok_emb + pos_emb # (B,T,C)
        
        tok_emb = self.transformer_blocks(tok_emb)  # transformer blocks
        tok_emb = self.ln_f(tok_emb)  # final layer norm
        
        logits = self.lm_head(tok_emb) 

        if targets == None:
            loss = None
        else:
            # reshape for pytorch
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indexes in the current context

        for _ in range(max_new_tokens):
            idx_cond = idx[:, -context_size:]  # crop to the last context_size tokens
            
            # get preds
            logits, loss = self(idx_cond)

            # only last step
            logits = logits[:, -1, :]  # (B, C)

            # probs
            probs = F.softmax(logits, dim=-1)  # (B, C)

            # sample from distribution (one prediction for each batch)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)

        return idx


m = BigramLanguageModel()
m.to(device)

logits, loss = m(xb, yb)
print(logits.shape)
print(loss)


print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long, device=device), max_new_tokens=100)[0].tolist()))


torch.Size([16384, 65])
tensor(4.3375, device='cuda:0', grad_fn=<NllLossBackward0>)

,ATto.q.UQxqjjTWVdRQN?. G!??YKH l!.EzsdnFS.RTneN.3,bnkgH
JMK VhrurR!;'rCOo3JovC-weIW!n:XE lAPMGP,VMo


In [75]:
#Train teh model 
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

for iter in range(max_iters):
    
    if iter % eval_interval == 0:
        print(f"step {iter}: train loss {loss.item():.4f}")
    
    # get batch
    xb, yb = get_batch("train")

    # forward pass
    logits, loss = m(xb, yb)

    # backward pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    # print("here")
    
print(loss.item())

step 0: train loss 4.3375
step 300: train loss 2.3333
step 600: train loss 1.9968
step 900: train loss 1.7251
step 1200: train loss 1.6285
step 1500: train loss 1.5367
step 1800: train loss 1.4508
step 2100: train loss 1.4065
step 2400: train loss 1.3538
step 2700: train loss 1.3074
step 3000: train loss 1.3210
step 3300: train loss 1.2686
step 3600: train loss 1.2719
step 3900: train loss 1.2313
step 4200: train loss 1.1988
step 4500: train loss 1.1736
step 4800: train loss 1.1699
1.184485912322998


In [78]:
#out from trained model
generated_text = decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long, device=device), max_new_tokens=10000)[0].tolist())
with open("output.txt", "w", encoding="utf-8") as out_file:
    out_file.write(generated_text)
